# Stage 3 – Campaign Outcome Simulation
Owner: _name_

One qubit per selected customer, RY(θ) with θ = 2·arcsin(√p) so P(|1⟩) = p. Success = at least 3 qubits read 1.

In [ ]:
import sys; sys.path.append('../src')
import numpy as np, pandas as pd
from config import *
np.random.seed(SEED)

## 1. Load the six selected customers and their Stage 1 probabilities

In [ ]:
s1 = pd.read_csv(STAGE1_RESULTS)
# TODO: replace with IDs from STAGE2_RESULTS once Stage 2 is done
selected_ids = s1.sort_values('rank')['customer_id'].head(6).tolist()  # placeholder
probs = s1.set_index('customer_id').loc[selected_ids, 'probability'].to_numpy()
pd.DataFrame({'qubit': range(6), 'customer_id': selected_ids, 'p': probs})

## 2. Exact answer (Poisson-binomial)

In [ ]:
def prob_at_least(probs, k):
    dist = np.array([1.0])
    for p in probs:
        dist = np.convolve(dist, [1 - p, p])
    return dist[k:].sum()
exact = prob_at_least(probs, 3)
exact

## 3. Quantum circuit
Qubit i ↔ selected_ids[i]. Note: Qiskit bitstrings are little-endian (qubit 0 is the rightmost character).

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

qc = QuantumCircuit(6)
for i, p in enumerate(probs):
    qc.ry(2 * np.arcsin(np.sqrt(p)), i)
qc.measure_all()

sim = AerSimulator(seed_simulator=SEED)
tqc = transpile(qc, sim, seed_transpiler=SEED)
counts = sim.run(tqc, shots=SHOTS).result().get_counts()
quantum_est = sum(c for b, c in counts.items() if b.count('1') >= 3) / SHOTS
print(qc.draw())
print('depth', tqc.depth(), 'ops', tqc.count_ops())

## 4. Monte Carlo check (classical)

In [ ]:
rng = np.random.default_rng(SEED)
mc = ((rng.random((SHOTS, 6)) < probs).sum(axis=1) >= 3).mean()
print(f'exact={exact:.4f}  quantum={quantum_est:.4f}  monte_carlo={mc:.4f}')

## 5. Save results

In [ ]:
pd.DataFrame([{'selected_ids': selected_ids, 'qubit_order': list(range(6)), 'probabilities': probs.tolist(),
               'shots': SHOTS, 'exact': exact, 'quantum_estimate': quantum_est, 'monte_carlo': mc,
               'abs_error_quantum': abs(quantum_est - exact)}]).to_csv(STAGE3_RESULTS, index=False)